In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

# Find the project root
current_dir = Path.cwd()

if (current_dir / "data" / "raw").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: c:\Users\shadr\Documents\orderflow-data-engineering
Data directory: c:\Users\shadr\Documents\orderflow-data-engineering\data\raw


In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Total CSV files: {len(csv_files)}")

for file in csv_files:
    print(file.name)

assert len(csv_files) == 9, "Expected 9 Olist CSV files."

Total CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
datasets = {}

for file in csv_files:

    name = file.stem.replace("olist_", "").replace("_dataset", "")

    df = pd.read_csv(file)

    datasets[name] = df

    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

customers: 99,441 rows, 5 columns
geolocation: 1,000,163 rows, 5 columns
order_items: 112,650 rows, 7 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 99,224 rows, 7 columns
orders: 99,441 rows, 8 columns
products: 32,951 rows, 9 columns
sellers: 3,095 rows, 4 columns
product_category_name_translation: 71 rows, 2 columns


In [4]:
summary = []

for name, df in datasets.items():

    summary.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Values": df.isna().sum().sum()
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,Table,Rows,Columns,Duplicate Rows,Missing Values
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,0,145903
5,orders,99441,8,0,4908
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,product_category_name_translation,71,2,0,0


In [5]:
for name in ["orders", "order_reviews", "products"]:

    df = datasets[name]

    missing = df.isna().sum()
    missing = missing[missing > 0]

    report = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": (missing / len(df) * 100).round(2)
    })

    print(f"\n{name.upper()}")
    display(report)


ORDERS


,Missing Count,Missing %
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98



ORDER_REVIEWS


,Missing Count,Missing %
review_comment_title,87656,88.34
review_comment_message,58247,58.70



PRODUCTS


,Missing Count,Missing %
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [6]:
geo = datasets["geolocation"]

print("Total records:", len(geo))
print("Exact duplicates:", geo.duplicated().sum())
print("Unique records:", len(geo.drop_duplicates()))

display(
    geo[geo.duplicated(keep=False)]
    .sort_values("geolocation_zip_code_prefix")
    .head(10)
)

Total records: 1000163
Exact duplicates: 261831
Unique records: 738332


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1004,1001,-23.549292,-46.633559,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
1435,1001,-23.549292,-46.633559,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
99,1001,-23.549292,-46.633559,sao paulo,SP
851,1001,-23.549825,-46.633970,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
1246,1001,-23.549292,-46.633559,sao paulo,SP


In [7]:
key_checks = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"]
}

results = []

for name, keys in key_checks.items():

    df = datasets[name]

    results.append({
        "Table": name,
        "Key": ", ".join(keys),
        "Rows": len(df),
        "Unique Keys": len(df.drop_duplicates(subset=keys)),
        "Null Key Rows": df[keys].isna().any(axis=1).sum(),
        "Duplicate Keys": df.duplicated(subset=keys).sum()
    })

key_report = pd.DataFrame(results)

display(key_report)

,Table,Key,Rows,Unique Keys,Null Key Rows,Duplicate Keys
0,customers,customer_id,99441,99441,0,0
1,orders,order_id,99441,99441,0,0
2,products,product_id,32951,32951,0,0
3,sellers,seller_id,3095,3095,0,0
4,order_items,"order_id, order_item_id",112650,112650,0,0
5,order_payments,"order_id, payment_sequential",103886,103886,0,0


In [8]:
relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]

fk_results = []

for child, child_key, parent, parent_key in relationships:

    child_df = datasets[child]
    parent_df = datasets[parent]

    orphan_count = (
        ~child_df[child_key].isin(parent_df[parent_key])
    ).sum()

    fk_results.append({
        "Child Table": child,
        "Parent Table": parent,
        "Foreign Key": child_key,
        "Orphan Records": int(orphan_count)
    })

display(pd.DataFrame(fk_results))

,Child Table,Parent Table,Foreign Key,Orphan Records
0,orders,customers,customer_id,0
1,order_items,orders,order_id,0
2,order_items,products,product_id,0
3,order_items,sellers,seller_id,0
4,order_payments,orders,order_id,0
5,order_reviews,orders,order_id,0
